In [ ]:
import pandas as pd
import re
import numpy as np

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

In [ ]:
df = pd.read_csv('instagram_usage_lifestyle.csv')
df.head()
df.shape

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

In [ ]:
clean_text("Hi **this is _me")

In [ ]:
df['text'] = (
    df['app_name'].astype(str) + " " +
    df['preferred_content_theme'].astype(str) + " " +
    df['content_type_preference'].astype(str)
)

In [ ]:
df['text'] = df['text'].apply(clean_text)

In [ ]:
y = df['user_engagement_score']

In [ ]:
y = (df['user_engagement_score'] > df['user_engagement_score'].median()).astype(int)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'],
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [ ]:
max_len = 50

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

In [ ]:
model = Sequential([
    Embedding(input_dim=5000, output_dim=64, input_length=max_len),
    LSTM(64, return_sequences=False),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

In [ ]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [ ]:
model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test_pad, y_test)
)

In [ ]:
loss, acc = model.evaluate(X_test_pad, y_test)
print("Accuracy:", acc)